# 04 - Split MS-ASL Universal CSV by has_video

This notebook does not download videos.
It only reads an existing MS-ASL CSV and creates:
- all rows,
- has_video == True rows,
- has_video == False rows.

In [5]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

In [6]:
PROJECT_ROOT = Path.cwd().resolve().parent.parent if Path.cwd().name == 'MS-ASL_EDA' else (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
MSASL_DIR = PROJECT_ROOT / 'MS-ASL'

INPUT_CSV = MSASL_DIR / 'msasl_universal_metadata_full_all_raw.csv'
OUTPUT_ALL_CSV = MSASL_DIR / 'msasl_universal_metadata_full_all_raw_rebuilt.csv'
OUTPUT_HAS_VIDEO_CSV = MSASL_DIR / 'msasl_universal_metadata_full_all_raw_has_video.csv'
OUTPUT_NO_VIDEO_CSV = MSASL_DIR / 'msasl_universal_metadata_full_all_raw_no_video.csv'

TARGET_COLUMNS = [
    'label',
    'source',
    'video_path',
    'start_frame',
    'end_frame',
    'length_frames',
    'duration_sec',
    'fps',
    'signer_id',
    'has_video',
    'video_width',
    'video_height',
]

print('PROJECT_ROOT:', PROJECT_ROOT)
print('INPUT_CSV exists:', INPUT_CSV.exists(), '|', INPUT_CSV)
print('OUTPUT_ALL_CSV:', OUTPUT_ALL_CSV)
print('OUTPUT_HAS_VIDEO_CSV:', OUTPUT_HAS_VIDEO_CSV)
print('OUTPUT_NO_VIDEO_CSV:', OUTPUT_NO_VIDEO_CSV)

PROJECT_ROOT: C:\Users\Magda\source\repos\private\szum
INPUT_CSV exists: True | C:\Users\Magda\source\repos\private\szum\MS-ASL\msasl_universal_metadata_full_all_raw.csv
OUTPUT_ALL_CSV: C:\Users\Magda\source\repos\private\szum\MS-ASL\msasl_universal_metadata_full_all_raw_rebuilt.csv
OUTPUT_HAS_VIDEO_CSV: C:\Users\Magda\source\repos\private\szum\MS-ASL\msasl_universal_metadata_full_all_raw_has_video.csv
OUTPUT_NO_VIDEO_CSV: C:\Users\Magda\source\repos\private\szum\MS-ASL\msasl_universal_metadata_full_all_raw_no_video.csv


In [7]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(f'Input CSV not found: {INPUT_CSV}')

df = pd.read_csv(INPUT_CSV)
print('Input rows:', len(df))
print('Input columns:', list(df.columns))

if 'has_video' not in df.columns:
    if 'download_status' in df.columns:
        df['has_video'] = df['download_status'].astype(str).str.lower().isin(['ok', 'already_exists'])
    else:
        raise ValueError('Input CSV must contain has_video (or download_status to infer it).')

if df['has_video'].dtype != bool:
    as_text = df['has_video'].astype(str).str.strip().str.lower()
    df['has_video'] = as_text.isin(['true', '1', 'yes'])

missing_columns = [c for c in TARGET_COLUMNS if c not in df.columns]
if missing_columns:
    raise ValueError(f'Missing required columns for universal schema: {missing_columns}')

df_all = df[TARGET_COLUMNS].copy()
df_has_video = df_all[df_all['has_video'] == True].copy()
df_no_video = df_all[df_all['has_video'] == False].copy()

df_has_video.to_csv(OUTPUT_HAS_VIDEO_CSV, index=False)
df_no_video.to_csv(OUTPUT_NO_VIDEO_CSV, index=False)

print('Saved has_video=True:', OUTPUT_HAS_VIDEO_CSV, '| rows =', len(df_has_video))
print('Saved has_video=False:', OUTPUT_NO_VIDEO_CSV, '| rows =', len(df_no_video))

Input rows: 25513
Input columns: ['label', 'source', 'video_path', 'start_frame', 'end_frame', 'length_frames', 'duration_sec', 'fps', 'signer_id', 'has_video', 'video_width', 'video_height']
Saved has_video=True: C:\Users\Magda\source\repos\private\szum\MS-ASL\msasl_universal_metadata_full_all_raw_has_video.csv | rows = 16816
Saved has_video=False: C:\Users\Magda\source\repos\private\szum\MS-ASL\msasl_universal_metadata_full_all_raw_no_video.csv | rows = 8697


In [8]:
print('has_video distribution:')
display(df_all['has_video'].value_counts(dropna=False).to_frame('count'))

print('Preview has_video=True:')
display(df_has_video.head(10))

print('Preview has_video=False:')
display(df_no_video.head(10))

has_video distribution:


,count
has_video,
True,16816
False,8697


Preview has_video=True:


,label,source,video_path,start_frame,end_frame,length_frames,duration_sec,fps,signer_id,has_video,video_width,video_height
0,match,msasl,MS-ASL/videos_MS_ASL_raw/msasl_raw_C37R_Ix8-qs...,0,83,84.0,2.767,30.000,0,True,640.0,360.0
1,fail,msasl,MS-ASL/videos_MS_ASL_raw/msasl_raw_PIsUJl8BN_I...,0,74,75.0,2.960,25.000,0,True,480.0,360.0
3,book,msasl,MS-ASL/videos_MS_ASL_raw/msasl_raw_J7tP98oDxqE...,0,66,67.0,2.640,25.000,0,True,480.0,360.0
4,sign language,msasl,MS-ASL/videos_MS_ASL_raw/msasl_raw_N2mG9ZKjrGA...,0,75,76.0,2.502,29.970,0,True,640.0,360.0
7,easter,msasl,MS-ASL/videos_MS_ASL_raw/msasl_raw_SVWABYmFdhs...,0,116,117.0,3.920,29.595,2,True,640.0,360.0
8,boring,msasl,MS-ASL/videos_MS_ASL_raw/msasl_raw_CYx7qm62Zwo...,0,71,72.0,2.840,25.000,13,True,640.0,360.0
10,phone,msasl,MS-ASL/videos_MS_ASL_raw/msasl_raw_HPz_C5XM4o4...,0,56,57.0,1.889,29.651,2,True,640.0,360.0
11,phone,msasl,MS-ASL/videos_MS_ASL_raw/msasl_raw_HPz_C5XM4o4...,73,123,51.0,1.686,29.651,2,True,640.0,360.0
12,phone,msasl,MS-ASL/videos_MS_ASL_raw/msasl_raw_HPz_C5XM4o4...,148,196,49.0,1.619,29.651,2,True,640.0,360.0
13,library,msasl,MS-ASL/videos_MS_ASL_raw/msasl_raw_S2cqitZ0qes...,0,73,74.0,2.920,25.000,0,True,480.0,360.0


Preview has_video=False:


,label,source,video_path,start_frame,end_frame,length_frames,duration_sec,fps,signer_id,has_video,video_width,video_height
2,laugh,msasl,NaN,0,31,32.0,1.034,29.970,4,False,640.0,360.0
5,school,msasl,NaN,33,110,78.0,2.569,29.970,1,False,640.0,360.0
6,school,msasl,NaN,140,206,67.0,2.203,29.970,1,False,640.0,360.0
9,past,msasl,NaN,0,32,33.0,1.068,29.970,191,False,1280.0,720.0
14,germany,msasl,NaN,0,36,37.0,1.201,29.970,4,False,640.0,360.0
15,like,msasl,NaN,0,52,53.0,1.735,29.970,269,False,640.0,360.0
29,portugal,msasl,NaN,0,44,45.0,1.468,29.970,4,False,640.0,360.0
54,yellow,msasl,NaN,10,60,51.0,1.668,29.970,1,False,640.0,360.0
55,yellow,msasl,NaN,90,160,71.0,2.336,29.970,1,False,640.0,360.0
64,again,msasl,NaN,0,117,118.0,3.900,30.003,27,False,640.0,360.0
